# Notebook 4: Embedding Generation

## Project: Enterprise Document Intelligence Assistant using LLM and RAG

This is the fourth notebook of the project.

In Notebook 1, we collected and standardized the BBC News dataset.

In Notebook 2, we cleaned and preprocessed the article text.

In Notebook 3, we split the documents into smaller text chunks.

In this notebook, we will convert each text chunk into a numerical vector using a sentence embedding model.

These embeddings will later be stored in a FAISS vector database for semantic search.

The goal of this notebook is to:

1. Load the chunked dataset from Notebook 3.
2. Generate embeddings for each text chunk.
3. Normalize embeddings for cosine similarity search.
4. Save embeddings as a NumPy file.
5. Save metadata separately for retrieval.

The output files from this notebook will be:

`bbc_chunk_embeddings.npy`

`bbc_chunk_metadata.csv`

These files will be used in:

`05_FAISS_Vector_Indexing.ipynb`

In [1]:
# Install required library
!pip install sentence-transformers -q

# Import required libraries
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer



# Find input file automatically
def find_input_file(file_name):
    """
    Searches for the input file in common Kaggle locations.

    This helps because files from previous notebooks must usually be
    uploaded as input datasets in Kaggle.
    """
    possible_paths = [
        file_name,
        f"/kaggle/working/{file_name}"
    ]

    # Search inside Kaggle input folders
    for root, dirs, files in os.walk("/kaggle/input"):
        if file_name in files:
            possible_paths.append(os.path.join(root, file_name))

    for path in possible_paths:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        f"{file_name} not found. Please upload it as input to this notebook."
    )


# Load chunked dataset
def load_chunked_dataset(file_name):
    """
    Loads the chunked dataset created in Notebook 3.
    """
    file_path = find_input_file(file_name)
    df = pd.read_csv(file_path)

    print("Chunked dataset loaded successfully.")
    print("Input file:", file_path)
    print("Dataset shape:", df.shape)

    return df


# Validate required columns
def validate_columns(df, required_columns):
    """
    Checks whether all required columns are available.
    """
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    print("All required columns are available.")


# Load embedding model
def load_embedding_model(model_name):
    """
    Loads a sentence-transformer embedding model.
    """
    model = SentenceTransformer(model_name)

    print("Embedding model loaded successfully.")
    print("Model name:", model_name)

    return model


# Generate embeddings
def generate_embeddings(texts, model, batch_size=64):
    """
    Converts text chunks into dense vector embeddings.

    Parameters:
        texts (list): List of chunk texts
        model: SentenceTransformer model
        batch_size (int): Number of texts processed at once

    Returns:
        numpy.ndarray: Embedding matrix
    """
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embeddings


# Save outputs
def save_outputs(embeddings, metadata, embedding_file, metadata_file):
    """
    Saves embeddings and metadata files.
    """
    np.save(embedding_file, embeddings)
    metadata.to_csv(metadata_file, index=False)

    print("Files saved successfully.")
    print("Embedding file:", embedding_file)
    print("Metadata file:", metadata_file)



In [2]:
# Load chunked dataset from Notebook 3
input_file = "bbc_docs_chunks.csv"

df_chunks = load_chunked_dataset(input_file)


# Preview input dataset
print("\nChunked Dataset Preview:")
display(df_chunks.head())

print("\nColumns:")
print(df_chunks.columns.tolist())


# Validate required columns
required_columns = [
    "chunk_id",
    "doc_id",
    "chunk_index",
    "title",
    "category",
    "chunk_text"
]

validate_columns(df_chunks, required_columns)

# Prepare text chunks for embedding
texts = df_chunks["chunk_text"].fillna("").astype(str).tolist()

print("\nNumber of chunks to embed:", len(texts))
print("Example chunk:\n")
print(texts[0][:700])


# Load embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = load_embedding_model(MODEL_NAME)


# Generate embeddings
embeddings = generate_embeddings(
    texts=texts,
    model=embedding_model,
    batch_size=64
)

print("\nEmbedding generation completed.")
print("Embedding matrix shape:", embeddings.shape)


# Create metadata file
metadata_columns = [
    "chunk_id",
    "doc_id",
    "chunk_index",
    "title",
    "category",
    "chunk_text"
]

if "chunk_word_count" in df_chunks.columns:
    metadata_columns.append("chunk_word_count")

df_metadata = df_chunks[metadata_columns].copy()

print("\nMetadata Preview:")
display(df_metadata.head())


# Check embedding example
print("\nSingle embedding vector example:")
print("Vector length:", len(embeddings[0]))
print("First 10 values:", embeddings[0][:10])


# Save embeddings and metadata
embedding_file = "bbc_chunk_embeddings.npy"
metadata_file = "bbc_chunk_metadata.csv"

save_outputs(
    embeddings=embeddings,
    metadata=df_metadata,
    embedding_file=embedding_file,
    metadata_file=metadata_file
)


# Verify saved files
loaded_embeddings = np.load(embedding_file)
loaded_metadata = pd.read_csv(metadata_file)

print("\nSaved files verified successfully.")
print("Loaded embeddings shape:", loaded_embeddings.shape)
print("Loaded metadata shape:", loaded_metadata.shape)

display(loaded_metadata.head())

Chunked dataset loaded successfully.
Input file: /kaggle/input/datasets/jahnavidulala/bbc-docs-chunks/bbc_docs_chunks.csv
Dataset shape: (8622, 7)

Chunked Dataset Preview:


,chunk_id,doc_id,chunk_index,title,category,chunk_text,chunk_word_count
0,1_1,1,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,21
1,2_1,2,1,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,31
2,3_1,3,1,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,21
3,4_1,4,1,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,20
4,5_1,5,1,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,20



Columns:
['chunk_id', 'doc_id', 'chunk_index', 'title', 'category', 'chunk_text', 'chunk_word_count']
All required columns are available.

Number of chunks to embed: 8622
Example chunk:

More than 1.5 million Ukrainians have fled the country. Here's what you need to know after day 11 of the war.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/135 [00:00<?, ?it/s]


Embedding generation completed.
Embedding matrix shape: (8622, 384)

Metadata Preview:


,chunk_id,doc_id,chunk_index,title,category,chunk_text,chunk_word_count
0,1_1,1,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,21
1,2_1,2,1,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,31
2,3_1,3,1,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,21
3,4_1,4,1,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,20
4,5_1,5,1,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,20



Single embedding vector example:
Vector length: 384
First 10 values: [ 0.09361183 -0.05255631 -0.00684102  0.00863639 -0.00140135 -0.00756832
  0.01082069 -0.04206086 -0.05888701 -0.01888965]
Files saved successfully.
Embedding file: bbc_chunk_embeddings.npy
Metadata file: bbc_chunk_metadata.csv

Saved files verified successfully.
Loaded embeddings shape: (8622, 384)
Loaded metadata shape: (8622, 7)


,chunk_id,doc_id,chunk_index,title,category,chunk_text,chunk_word_count
0,1_1,1,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,21
1,2_1,2,1,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,31
2,3_1,3,1,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,21
3,4_1,4,1,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,20
4,5_1,5,1,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,20
